# Obtención de números de batch para casos en Target List

Este notebook procesa el archivo `Target_List_Cluster.txt` para añadir, por cada línea, los números de batch (dos primeros dígitos) correspondientes a los pacientes listados. El proceso sigue estos pasos:

1. Extraer los identificadores de caso (C3L-*, C3N-*) desde el Target List.
2. Consultar la API de GDC para obtener el mapping de cada caso a sus alícuotas (aliquot_submitter_id).
3. Cargar el archivo `Mapa_Pacientes_CPTAC.txt` que contiene la relación entre identificadores (CPT o C3L) y el batch (carpeta) al que pertenecen.
4. Asignar a cada caso su batch mediante coincidencia directa (para IDs de caso) o por prefijo de alícuota (para IDs CPT).
5. Generar un nuevo archivo Target List con una columna adicional que contiene los números de batch (separados por coma si hay varios).

## 1. Configuración y constantes

Define las rutas de los archivos de entrada y salida. Todos los archivos generados se guardarán en el directorio actual.

In [3]:
import requests
import pandas as pd
import json
import re
import ast
from collections import defaultdict

# Rutas de archivos (ajustar si es necesario)
TARGET_LIST_PATH = "../data/interim/proteomal/Target_List_Cluster.txt"
BATCH_MAP_PATH = "../data/interim/proteomal/Mapa_Pacientes_CPTAC.txt"

# Archivos de salida
ALIQUOT_MAPPING_OUT = "../data/interim/proteomal/case_aliquot_mapping.csv"
BATCH_ASSIGNMENT_DETAILS = "../data/interim/proteomal/batch_assignment_details.csv"
CASES_WITH_BATCH_OUT = "../data/interim/proteomal/cases_with_batch.csv"
FINAL_TARGET_LIST_OUT = "../data/interim/proteomal/Target_List_Cluster_with_batch_numbers.csv"

## 2. Extraer casos de interés desde Target List

Lee el archivo Target List (formato separado por `|`) y extrae todos los identificadores de caso que comienzan con `C3L-` o `C3N-`.

In [4]:
def extract_case_ids_from_target(file_path):
    """Lee el archivo Target List y devuelve una lista ordenada de case_ids únicos."""
    case_set = set()
    with open(file_path, 'r') as f:
        next(f)  # saltar cabecera
        for line in f:
            parts = line.strip().split('|')
            if len(parts) >= 3:
                pacientes = parts[2].split(', ')
                for p in pacientes:
                    p = p.strip()
                    if p.startswith(('C3L-', 'C3N-')):
                        case_set.add(p)
    return sorted(case_set)

case_ids = extract_case_ids_from_target(TARGET_LIST_PATH)
print(f"Total de casos únicos en Target List: {len(case_ids)}")
print("Primeros 10:", case_ids[:10])

Total de casos únicos en Target List: 140
Primeros 10: ['C3L-00017', 'C3L-00102', 'C3L-00189', 'C3L-00277', 'C3L-00401', 'C3L-00589', 'C3L-00598', 'C3L-00599', 'C3L-00622', 'C3L-00625']


## 3. Obtener mapping de casos a alícuotas desde la API de GDC

Consulta la API de GDC para cada caso y obtiene los `aliquot_submitter_id` asociados. El resultado se guarda en `case_aliquot_mapping.csv`.

In [5]:
# Construir filtro para la consulta
filters = {
    "op": "in",
    "content": {
        "field": "submitter_id",
        "value": case_ids
    }
}

params = {
    "filters": json.dumps(filters),
    "fields": "submitter_id,samples.portions.analytes.aliquots.submitter_id",
    "format": "JSON",
    "size": len(case_ids)
}

print("Consultando API de GDC...")
response = requests.post("https://api.gdc.cancer.gov/cases", json=params)

if response.status_code != 200:
    raise Exception(f"Error en consulta GDC: {response.status_code}\n{response.text}")

data = response.json()
hits = data.get('data', {}).get('hits', [])

# Extraer mapping caso -> alícuota
mapping = []
for case in hits:
    case_submitter = case.get('submitter_id')
    for sample in case.get('samples', []):
        for portion in sample.get('portions', []):
            for analyte in portion.get('analytes', []):
                for aliquot in analyte.get('aliquots', []):
                    aliquot_submitter = aliquot.get('submitter_id')
                    if aliquot_submitter:
                        mapping.append({
                            'case_id': case_submitter,
                            'aliquot_submitter_id': aliquot_submitter
                        })

df_mapping = pd.DataFrame(mapping)
df_mapping.to_csv(ALIQUOT_MAPPING_OUT, index=False)
print(f"Mapping guardado: {len(df_mapping)} filas en {ALIQUOT_MAPPING_OUT}")
print(df_mapping.head())

Consultando API de GDC...
Mapping guardado: 844 filas en ../data/interim/proteomal/case_aliquot_mapping.csv
     case_id aliquot_submitter_id
0  C3N-02592        CPT0162760009
1  C3N-02592        CPT0162760006
2  C3N-02592        CPT0162820002
3  C3N-02768        CPT0208030002
4  C3N-02768        CPT0207980008


## 4. Cargar mapa de batches (Mapa_Pacientes_CPTAC.txt)

Este archivo contiene líneas con el formato: `ruta|id_raw`. De cada línea extraemos:
- `id_real`: el identificador (puede ser CPT… o C3L-…)
- `batch`: el nombre de la carpeta (primer componente de la ruta)

In [6]:
batch_list = []
with open(BATCH_MAP_PATH, 'r') as f:
    for line in f:
        line = line.strip()
        if '|' not in line:
            continue
        parts = line.split('|')
        if len(parts) < 2:
            continue
        ruta = parts[0].strip()
        id_raw = parts[1].strip()
        # Limpiar ruta (quitar './' inicial)
        if ruta.startswith('./'):
            ruta = ruta[2:]
        batch = ruta.split('/')[0]
        # Separar prefijo (ej. 127C) del ID real si existe
        id_parts = id_raw.split()
        if len(id_parts) == 2:
            id_real = id_parts[1]
        else:
            id_real = id_raw
        batch_list.append({'id_real': id_real, 'batch': batch})

df_batch = pd.DataFrame(batch_list).drop_duplicates()
print(f"Mapa de batches cargado: {len(df_batch)} entradas únicas")
print(df_batch.head(10))

Mapa de batches cargado: 235 entradas únicas
         id_real                              batch
0  CPT0237840004  01CPTAC_PDA_Proteome_JHU_20191125
1  CPT0063990003  01CPTAC_PDA_Proteome_JHU_20191125
2  CPT0166780004  01CPTAC_PDA_Proteome_JHU_20191125
3   C3L-01687-04  01CPTAC_PDA_Proteome_JHU_20191125
4  CPT0166760004  01CPTAC_PDA_Proteome_JHU_20191125
5  CPT0094630004  01CPTAC_PDA_Proteome_JHU_20191125
6  CPT0063970004  01CPTAC_PDA_Proteome_JHU_20191125
7  CPT0237850004  01CPTAC_PDA_Proteome_JHU_20191125
8  CPT0081260004  01CPTAC_PDA_Proteome_JHU_20191125
9  CPT0123990003  02CPTAC_PDA_Proteome_JHU_20191022


## 5. Asignar batches a casos mediante prefijos de alícuotas

Estrategia:
- Para cada caso, obtenemos todas sus alícuotas (del mapping de GDC) y extraemos un **prefijo** (primeros 10 caracteres de los IDs CPT, p.ej. `CPT0094630`).
- Construimos un diccionario que mapea cada prefijo al caso correspondiente.
- Recorremos el mapa de batches:
  - Si `id_real` es un caso (C3L/C3N), lo normalizamos (quitamos sufijo `-XX`) y lo asignamos directamente.
  - Si `id_real` es un CPT, buscamos su prefijo en el diccionario para encontrar el caso.
- Al final, para cada caso obtenemos el conjunto de batches asociados (normalmente uno).

In [7]:
# Funciones auxiliares
def normalize_aliquot(aliq):
    """Elimina sufijos tipo _1, _2 del final del aliquot_id."""
    return re.sub(r'_\d+$', '', str(aliq))

def extract_prefix(aliq):
    """Para IDs CPT, devuelve los primeros 10 caracteres (CPT + 7 dígitos)."""
    if aliq.startswith('CPT'):
        return aliq[:10]
    return None

def normalize_case_id(cid):
    """Convierte C3L-01687-04 a C3L-01687."""
    match = re.match(r'(C3[LN]-\d+)(?:-\d+)?$', str(cid))
    return match.group(1) if match else cid

# 1. Preparar mapping de casos a prefijos
df_mapping = pd.read_csv(ALIQUOT_MAPPING_OUT)
df_mapping['aliquot_norm'] = df_mapping['aliquot_submitter_id'].apply(normalize_aliquot)

prefijos_por_caso = defaultdict(set)
for _, row in df_mapping.iterrows():
    case = row['case_id']
    aliq = row['aliquot_norm']
    pref = extract_prefix(aliq)
    if pref:
        prefijos_por_caso[case].add(pref)

# Construir diccionario inverso: prefijo -> caso (asumiendo que cada prefijo pertenece a un solo caso)
prefijo_a_caso = {}
for case, prefijos in prefijos_por_caso.items():
    for pref in prefijos:
        if pref in prefijo_a_caso:
            print(f"Advertencia: prefijo {pref} ya asignado a {prefijo_a_caso[pref]}, también aparece en {case}")
        else:
            prefijo_a_caso[pref] = case

print(f"Prefijos únicos generados: {len(prefijo_a_caso)}")

# 2. Asignar batches a cada id_real del mapa
asignaciones = []  # cada elemento: id_real, case_id, batch, metodo

for _, row in df_batch.iterrows():
    id_real = row['id_real']
    batch = row['batch']
    
    if id_real.startswith(('C3L-', 'C3N-')):
        # ID de caso: normalizar y asignar directamente
        case_norm = normalize_case_id(id_real)
        asignaciones.append((id_real, case_norm, batch, 'caso_directo'))
    elif id_real.startswith('CPT'):
        # ID de alícuota: buscar por prefijo
        pref = id_real[:10]
        case_asignado = prefijo_a_caso.get(pref)
        if case_asignado:
            asignaciones.append((id_real, case_asignado, batch, 'prefijo'))
        else:
            # No se encontró caso para este prefijo
            asignaciones.append((id_real, None, batch, 'no_asignado'))
    else:
        # Otros formatos (no deberían aparecer)
        asignaciones.append((id_real, None, batch, 'desconocido'))

df_asignacion = pd.DataFrame(asignaciones, columns=['id_real', 'case_id', 'batch', 'metodo'])
print("\nResumen de asignaciones:")
print(df_asignacion['metodo'].value_counts())

# Guardar detalle
df_asignacion.to_csv(BATCH_ASSIGNMENT_DETAILS, index=False)
print(f"Detalle guardado en {BATCH_ASSIGNMENT_DETAILS}")

# 3. Consolidar por caso: obtener el batch (o batches) para cada caso
# Filtrar solo asignaciones exitosas
df_ok = df_asignacion[df_asignacion['case_id'].notna()]

# Agrupar por caso y recoger batches únicos
casos_batch = df_ok.groupby('case_id')['batch'].agg(lambda x: list(set(x))).reset_index()
casos_batch.rename(columns={'batch': 'batches'}, inplace=True)

# Verificar si algún caso tiene más de un batch
multi_batch = casos_batch[casos_batch['batches'].apply(len) > 1]
if not multi_batch.empty:
    print(f"\nAtención: {len(multi_batch)} casos tienen múltiples batches:")
    print(multi_batch)
    # Para estos casos, podríamos tomar el primer batch o concatenar. Aquí tomamos el primero.
    casos_batch['batch_unico'] = casos_batch['batches'].apply(lambda x: x[0])
else:
    casos_batch['batch_unico'] = casos_batch['batches'].apply(lambda x: x[0])

# Guardar mapping caso -> batch único
casos_batch[['case_id', 'batch_unico']].to_csv(CASES_WITH_BATCH_OUT, index=False)
print(f"\nMapping caso->batch guardado en {CASES_WITH_BATCH_OUT}")
print("Primeros 10 casos:")
print(casos_batch[['case_id', 'batch_unico']].head(10))

Prefijos únicos generados: 370

Resumen de asignaciones:
metodo
prefijo         201
no_asignado      30
caso_directo      4
Name: count, dtype: int64
Detalle guardado en ../data/interim/proteomal/batch_assignment_details.csv

Mapping caso->batch guardado en ../data/interim/proteomal/cases_with_batch.csv
Primeros 10 casos:
     case_id                        batch_unico
0  C3L-00017  20CPTAC_PDA_Proteome_JHU_20200121
1  C3L-00102  16CPTAC_PDA_Proteome_JHU_20200103
2  C3L-00189  14CPTAC_PDA_Proteome_JHU_20191227
3  C3L-00277  11CPTAC_PDA_Proteome_JHU_20191211
4  C3L-00401  23CPTAC_PDA_Proteome_JHU_20200203
5  C3L-00589  01CPTAC_PDA_Proteome_JHU_20191125
6  C3L-00598  17CPTAC_PDA_Proteome_JHU_20200110
7  C3L-00599  01CPTAC_PDA_Proteome_JHU_20191125
8  C3L-00622  21CPTAC_PDA_Proteome_JHU_20200124
9  C3L-00625  14CPTAC_PDA_Proteome_JHU_20191227


## 6. Generar archivo final con números de batch por línea de Target List

Cargamos el mapping caso → batch (con el nombre completo de la carpeta) y extraemos solo los dos primeros dígitos (número de batch). Luego recorremos el Target List original y, para cada línea, agregamos una columna con los números de batch de los pacientes listados (ordenados y separados por coma).

In [8]:
# Cargar mapping de caso a batch (carpeta completa)
df_casos = pd.read_csv(CASES_WITH_BATCH_OUT)

# Extraer número de batch (primeros dos dígitos)
def extraer_numero(carpeta):
    match = re.match(r'^(\d{2})', carpeta)
    return match.group(1) if match else carpeta

df_casos['numero_batch'] = df_casos['batch_unico'].apply(extraer_numero)
caso_a_numero = dict(zip(df_casos['case_id'], df_casos['numero_batch']))

# Leer Target List original
with open(TARGET_LIST_PATH, 'r') as f:
    lines = f.readlines()

# Procesar cabecera y líneas de datos
header = lines[0].strip()
new_header = header + '|Batch_Numeros'
result_lines = [new_header]

for line in lines[1:]:
    line = line.strip()
    if not line:
        continue
    parts = line.split('|')
    if len(parts) < 3:
        continue
    gen = parts[0]
    peptido = parts[1]
    pacientes_str = parts[2]
    pacientes = [p.strip() for p in pacientes_str.split(',')]
    
    # Recolectar números de batch únicos
    numeros = set()
    for p in pacientes:
        if p in caso_a_numero:
            numeros.add(caso_a_numero[p])
        else:
            print(f"Advertencia: paciente {p} no encontrado en mapping")
    
    numeros_ordenados = sorted(numeros, key=lambda x: int(x))
    numeros_str = ','.join(numeros_ordenados)
    
    new_line = f"{gen}|{peptido}|{pacientes_str}|{numeros_str}"
    result_lines.append(new_line)

# Guardar resultado
with open(FINAL_TARGET_LIST_OUT, 'w') as f:
    f.write('\n'.join(result_lines))

print(f"\nArchivo final generado: {FINAL_TARGET_LIST_OUT}")
print("Primeras líneas:")
with open(FINAL_TARGET_LIST_OUT, 'r') as f:
    for i, line in enumerate(f):
        if i >= 5: break
        print(line.strip())


Archivo final generado: ../data/interim/proteomal/Target_List_Cluster_with_batch_numbers.csv
Primeras líneas:
Gen|Peptido_Mutado|Pacientes_CPTAC|Batch_Numeros
MUC4|ATPLPVTSL|C3L-01054, C3L-02899, C3L-04080, C3L-04848, C3L-04853, C3N-00511, C3N-01167, C3N-01168, C3N-01381, C3N-01388, C3N-01715, C3N-02971, C3N-03061, C3N-03670|02,04,06,08,11,14,16,19,20,22,23,24
HYDIN|KPRNVQVFF|C3L-00589, C3L-01037, C3L-02116, C3N-00302, C3N-00511, C3N-00514, C3N-01715, C3N-03754, C3N-03840|01,07,12,15,16,17,20,22,24
NMRAL1|TTFIVTNYW|C3N-01715|22
MUC6|MPTGTIPPL|C3L-01689|04


## 7. Resumen final

Se ha generado el archivo `Target_List_Cluster_with_batch_numbers.csv` que contiene la misma información que el original más una columna adicional `Batch_Numeros` con los números de batch (dos dígitos) de los pacientes involucrados en cada línea.